# Radon Analysis

Evaluating the radon mitigation system in a first-floor apartment unit (above a garage) from hourly sensor readings. The key findings are summarized first; system background, full timeline, and data-quality notes follow.

## Summary of findings

**Bottom line.** The mitigation system — a fan that depressurizes a column running down to the garage below this first-floor unit — lowers the *typical* indoor radon level but does **not** prevent recurring spikes at or above the EPA action level of 4.0 pCi/L. Excursions continue through the most recent reading and after the latest system adjustment. Keeping that radon out is the system's purpose; that goal is not yet met.

**The evidence** — 532 hourly readings, 2026-05-14 → 06-12 (post-relocation, directly comparable):

- **Frequent:** 22 hours (4.1%) reached the EPA level and 40 (7.5%) the WHO level (2.7); at least one EPA-level hour on **10 of 30 days — about one in three.**
- **Large:** **12 distinct EPA-exceeding events, peaking at 15.0 pCi/L — nearly 4× the action level.**
- **Sustained:** 8 of the 12 events lasted ≥ 2 consecutive hours; two lasted three.
- **Not improving:** in the 20 days after the final adjustment (2026-05-23), the EPA level was still reached in 6 events / 12 hours — and the period's largest spike, **14.0 pCi/L, is the most recent reading (evening of June 12).**
- **Predictable:** **68% of EPA-exceeding hours fall between 19:00 and 22:00.**

**A prior configuration did better in this same unit.** The temporary system that ran here for years held a 90th-percentile near 1.2 pCi/L and a steady-operation maximum near 2.7 — below the action level. The current permanent system differs in only one intended way: **the fan was moved farther from the suction points** (vacuum later raised 2.5" → 3.5" WC to compensate). It matches the old *typical* level but now allows 4–15 pCi/L spikes the old one did not — a regression in exactly the excursions a mitigation system exists to prevent. The configuration-by-configuration comparison is in the next section.

*Why not just cite the average? EPA's 4.0 pCi/L is an annual-average screening benchmark, not a license for individual hours at 10–15. These late-spring readings are also a seasonal floor (winter is typically worse), and windows were opened during some high readings — so the peaks here are likely underestimates.*

## Background & system history

First floor of an apartment building, directly above a garage; radon enters from below. The mitigation system is a fan that depressurizes a vertical column connecting down to the garage — **not** a sub-slab system. Reference levels (pCi/L): **EPA 4.0**, **WHO 2.7**, US indoor average 1.3.

**Performance by configuration** — median, 90th percentile, and maximum hourly reading. The temporary-system figures are from the 2023 commissioning dataset; the current permanent system is split into its three 2026 adjustment phases:

| Configuration | Median | p90 | Max |
|---|---:|---:|---:|
| No system (2023) | 12.4 | ~21 | ~58 |
| V1 — temporary, 1 suction point (2023) | 2.9 | 6.3 | 8.8 |
| **V2 — temporary, 3 suction points (2023)** | **0.4** | **1.2** | **2.7** ¹ |
| V3 permanent · P1 — fan + 2.5" WC (May 14–18) | 1.3 | 4.8 | 15.0 |
| V3 permanent · P2 — 3.5" WC + sealing (May 18–23) | 0.8 | 1.4 | 3.8 |
| **V3 permanent · P3 — + ext. tweaks (May 23 → now)** | **0.6** | **1.3** | **14.0** |

¹ V2's maximum was ~2.7 in steady operation; a single 8.6 reading occurred at the install timestamp and settled within the hour.

The **V2 temporary configuration** ran for several years and never reached the EPA action level in steady operation. The current **permanent system (V3)** is the same three-suction-point design with the **fan relocated farther from the suction points** — the only intended change. Its typical level and p90 now roughly match V2, but its **tail does not**: V3 repeatedly reaches 4–15 pCi/L where V2 held near 2.7.

**Timeline (2026):** 05-12 permanent system commissioned · 05-14 sensor moved to current location (only later readings are comparable — it was previously in a lower-radon room) · 05-18 vacuum 2.5" → 3.5" WC + interior sealing · 05-23 external suction-point adjustments.

**Notes:** all statistics use post-05-14 readings for comparability; spike peaks may be underestimated (windows opened during some high readings); late-spring data represents a seasonal low.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

EPA_LEVEL = 4.0
WHO_LEVEL = 2.7

SENSOR_MOVED = '2026-05-14'  # all analysis restricted to this date forward

# (date, short label, color) — mitigation events in green, sensor event in purple
EVENTS = [
    ('2026-05-12', 'Fan',      'green'),
    ('2026-05-14', 'Sensor →', 'purple'),
    ('2026-05-18', '3.5" WC',  'green'),
    ('2026-05-23', 'Ext.',     'green'),
]

# Phases for comparison (start inclusive, end exclusive; end=None means open-ended)
PHASES = [
    ('P1: fan + 2.5" WC',    '2026-05-14', '2026-05-18'),
    ('P2: 3.5" WC + sealing', '2026-05-18', '2026-05-23'),
    ('P3: + ext. tweaks',     '2026-05-23', None),
]

## Load data

In [2]:
df = pd.read_csv(
    '04-22-2026_06-12-2026-export.csv',
    encoding='utf-8-sig',  # strip the UTF-8 BOM on the header row
    parse_dates=['Measurement Time'],
    date_format='%m/%d/%Y %H:%M',
)
df = df.rename(columns={'Measurement Time': 'ts', 'Radon Hourly Reading': 'radon'}).set_index('ts').sort_index()

df_post = df.loc[SENSOR_MOVED:].copy()  # comparable readings only

print(f'Full dataset:     {len(df):>5} rows  |  {df.index.min()} → {df.index.max()}')
print(f'Post-sensor-move: {len(df_post):>5} rows  |  {df_post.index.min()} → {df_post.index.max()}')
df.head()

Full dataset:       918 rows  |  2026-04-25 19:24:00 → 2026-06-12 20:36:00
Post-sensor-move:   532 rows  |  2026-05-14 00:25:00 → 2026-06-12 20:36:00


,radon
ts,
2026-04-25 19:24:00,1.2
2026-04-25 20:24:00,0.9
2026-04-25 21:24:00,0.4
2026-04-25 22:24:00,0.5
2026-04-25 23:24:00,0.4


## Helper: chart annotations

Uses `add_shape` + `add_annotation` directly rather than `add_hline/vline/vrect`'s `annotation_text=` parameter, to sidestep a plotly bug that trips on pandas 3.0 Timestamp arithmetic.

In [3]:
EVENTS_POST = [(d, l, c) for d, l, c in EVENTS if pd.Timestamp(d) >= pd.Timestamp(SENSOR_MOVED)]


def annotate(fig, *, levels=True, events=None, shade_pre_move=False, height=600):
    """Add EPA/WHO horizontal lines, event vertical lines, and optional pre-move shading."""
    fig.update_layout(height=height)
    if levels:
        for y, color, dash, name in [(EPA_LEVEL, 'red', 'dash', 'EPA'),
                                     (WHO_LEVEL, 'orange', 'dot', 'WHO')]:
            fig.add_shape(type='line', xref='x domain', yref='y',
                          x0=0, x1=1, y0=y, y1=y,
                          line=dict(color=color, dash=dash, width=1))
            fig.add_annotation(xref='x domain', yref='y',
                               x=1, y=y, text=f'{name} {y}', showarrow=False,
                               xanchor='right', yanchor='bottom',
                               font=dict(size=10, color=color))
    if events:
        for i, (date, label, color) in enumerate(events):
            x = pd.Timestamp(date)
            fig.add_shape(type='line', xref='x', yref='y domain',
                          x0=x, x1=x, y0=0, y1=1,
                          line=dict(color=color, dash='dash', width=1.5))
            on_top = (i % 2 == 0)
            fig.add_annotation(xref='x', yref='y domain',
                               x=x, y=1.02 if on_top else -0.04,
                               text=label, showarrow=False,
                               xanchor='center',
                               yanchor='bottom' if on_top else 'top',
                               font=dict(size=10, color=color))
    if shade_pre_move:
        fig.add_shape(type='rect', xref='x', yref='y domain',
                      x0=df.index.min(), x1=pd.Timestamp(SENSOR_MOVED),
                      y0=0, y1=1,
                      fillcolor='gray', opacity=0.18, line_width=0, layer='below')
        fig.add_annotation(xref='x', yref='y domain',
                           x=df.index.min(), y=0.97,
                           text='not comparable (sensor in other room)',
                           showarrow=False,
                           xanchor='left', yanchor='top',
                           font=dict(size=10, color='gray'))
    return fig

## Raw hourly time series (full dataset)

Shaded region = sensor was in a different room; values are not comparable to post-move readings.

In [4]:
fig = px.line(df, y='radon', title='Hourly radon (pCi/L) — full dataset',
              labels={'radon': 'pCi/L', 'ts': ''})
fig.update_traces(line=dict(width=1))
annotate(fig, events=EVENTS, shade_pre_move=True)
fig.show()

---

# Post-sensor-move analysis

Everything below uses **`df_post`** — readings from 2026-05-14 onward only.

In [5]:
df_post['radon'].describe().round(2)

count    532.00
mean       1.09
std        1.47
min        0.20
25%        0.50
50%        0.70
75%        1.00
max       15.00
Name: radon, dtype: float64

## Daily distribution (percentiles)

For each day: the **median (p50)** line shows the typical level; the **IQR band (p25–p75)** shows the middle half of readings; **p90/p95** capture the upper tail; the **daily max** marker is the single worst hour. A day where median is low but max hits EPA is a day where the system is mostly working but failed at least once.

In [6]:
daily = pd.DataFrame({
    'p25':       df_post['radon'].resample('D').quantile(0.25),
    'p50':       df_post['radon'].resample('D').quantile(0.50),
    'p75':       df_post['radon'].resample('D').quantile(0.75),
    'p90':       df_post['radon'].resample('D').quantile(0.90),
    'p95':       df_post['radon'].resample('D').quantile(0.95),
    'daily_max': df_post['radon'].resample('D').max(),
})

fig = go.Figure()
# IQR band (p25-p75) — invisible upper bound, then fill down from p25
fig.add_trace(go.Scatter(x=daily.index, y=daily['p75'],
                         line=dict(color='rgba(0,0,0,0)'), mode='lines',
                         showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=daily.index, y=daily['p25'], name='IQR (p25–p75)',
                         line=dict(color='rgba(0,0,0,0)'), mode='lines',
                         fill='tonexty', fillcolor='rgba(70, 130, 180, 0.20)'))
# percentile lines
fig.add_trace(go.Scatter(x=daily.index, y=daily['p50'], name='p50 (median)',
                         line=dict(color='steelblue', width=2.5)))
fig.add_trace(go.Scatter(x=daily.index, y=daily['p90'], name='p90',
                         line=dict(color='goldenrod', width=2)))
fig.add_trace(go.Scatter(x=daily.index, y=daily['p95'], name='p95',
                         line=dict(color='darkorange', width=2)))
fig.add_trace(go.Scatter(x=daily.index, y=daily['daily_max'], name='daily max',
                         mode='markers',
                         marker=dict(color='crimson', size=9, symbol='diamond')))
fig.update_layout(title='Daily radon distribution: median, IQR, upper percentiles, and max',
                  yaxis_title='pCi/L', xaxis_title='')
annotate(fig, events=EVENTS_POST)
fig.show()

## 24-hour rolling: mean vs. p90

The 24-hour rolling mean (blue) shows the typical level over the prior day. The 24-hour rolling 90th percentile (red, dashed) shows what level the worst 10% of hours in that window hit. The gap between them is informative: a low rolling mean alongside a high rolling p90 indicates the system is "mostly working but occasionally failing" — a failure mode that a mean-only chart hides.

Hourly readings are shown in light gray for context. The 7-day window is omitted here because the post-move dataset spans under a month; daily-level smoothing is already covered in the daily-distribution chart above.

In [7]:
window_24h = df_post['radon'].rolling('24h')

ma = pd.DataFrame({
    'hourly':  df_post['radon'],
    '24h mean': window_24h.mean(),
    '24h p90':  window_24h.quantile(0.9),
})

fig = px.line(ma, title='24-hour rolling: mean vs. p90',
              labels={'value': 'pCi/L', 'ts': '', 'variable': ''})
fig.update_traces(selector=dict(name='hourly'),   line=dict(color='lightgray', width=1))
fig.update_traces(selector=dict(name='24h mean'), line=dict(color='steelblue', width=2.5))
fig.update_traces(selector=dict(name='24h p90'),  line=dict(color='crimson', width=2.5, dash='dash'))
annotate(fig, events=EVENTS_POST)
fig.show()

## Hour-of-day pattern

Radon often peaks overnight (less air exchange) and dips during the day. If mitigation is effective and running 24/7, you'd expect this pattern to flatten.

In [8]:
by_hour = df_post.copy()
by_hour['hour'] = by_hour.index.hour

fig = px.box(by_hour, x='hour', y='radon', points=False,
             title='Radon distribution by hour of day (post-sensor-move)',
             labels={'radon': 'pCi/L', 'hour': 'hour of day'})
annotate(fig)
fig.show()

## Evening-only timeline (19:00–22:00)

The hour-of-day boxplot above shows readings peak in the evening. This filters the dataset down to just those hours so the temporal pattern is visible across phases: how does the evening profile in P1 (early mitigation), P2 (after the May 18 sealing + 3.5" WC change), and P3 (after the May 23 external adjustments) compare? Each dot is one evening hourly reading.

In [9]:
evening = df_post[df_post.index.hour.isin([19, 20, 21, 22])].copy()

fig = px.scatter(evening, y='radon',
                 title='Evening hours only (19:00–22:00) — the spike risk window',
                 labels={'radon': 'pCi/L', 'ts': ''})
fig.update_traces(marker=dict(size=8, color='steelblue', opacity=0.7))
annotate(fig, events=EVENTS_POST)
fig.show()

## Day-of-week pattern

In [10]:
by_dow = df_post.copy()
by_dow['dow'] = by_dow.index.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig = px.box(by_dow, x='dow', y='radon', points=False,
             category_orders={'dow': dow_order},
             title='Radon distribution by day of week (post-sensor-move)',
             labels={'radon': 'pCi/L', 'dow': ''})
annotate(fig)
fig.show()

## Threshold exceedance — headline metrics

Three counts that **overlap by design**: hours below the WHO recommendation, hours at or above WHO, and hours at or above EPA. Every hour above EPA is also above WHO, so these don't sum to 100% — they're three independent answers to "how often was radon at concerning levels?" The next chart breaks the same exceedance counts down day by day.

In [11]:
total = len(df_post)
below_who    = int((df_post['radon'] <  WHO_LEVEL).sum())
at_above_who = int((df_post['radon'] >= WHO_LEVEL).sum())
at_above_epa = int((df_post['radon'] >= EPA_LEVEL).sum())

cards = [
    ('Below WHO recommendation', f'< {WHO_LEVEL} pCi/L', below_who,    'steelblue'),
    ('At or above WHO',          f'≥ {WHO_LEVEL} pCi/L', at_above_who, 'goldenrod'),
    ('At or above EPA',          f'≥ {EPA_LEVEL} pCi/L', at_above_epa, 'crimson'),
]

fig = make_subplots(rows=1, cols=3,
                    specs=[[{'type': 'indicator'}] * 3])

for col, (label, sub, n, color) in enumerate(cards, start=1):
    pct = n / total * 100
    fig.add_trace(go.Indicator(
        mode='number',
        value=n,
        number={
            'font': {'color': color, 'size': 64},
            'suffix': f'<br><span style="font-size:0.40em;color:#666">'
                      f'{pct:.1f}% of hours'
                      f'</span>',
        },
        title={
            'text': f'<b>{label}</b><br>'
                    f'<span style="font-size:0.85em;color:#666">{sub}</span>',
            'font': {'size': 14},
        },
    ), row=1, col=col)

fig.update_layout(height=260, margin=dict(t=0, b=20, l=20, r=20))
fig.show()

## Hours per day above thresholds

Hours per day at or above each threshold. Any bar > 0 on the EPA series is a day where the mitigation system did not keep radon below the action level for the full day.

In [12]:
hours_above_who_daily = (df_post['radon'] >= WHO_LEVEL).resample('D').sum()
hours_above_epa_daily = (df_post['radon'] >= EPA_LEVEL).resample('D').sum()

fig = go.Figure()
fig.add_trace(go.Bar(x=hours_above_who_daily.index, y=hours_above_who_daily.values,
                     name=f'≥ WHO ({WHO_LEVEL})', marker_color='orange', opacity=0.55))
fig.add_trace(go.Bar(x=hours_above_epa_daily.index, y=hours_above_epa_daily.values,
                     name=f'≥ EPA ({EPA_LEVEL})', marker_color='crimson'))
fig.update_layout(title='Hours per day above radon thresholds',
                  yaxis_title='hours per day', xaxis_title='', barmode='overlay')
annotate(fig, levels=False, events=EVENTS_POST)
fig.show()

## Spike events (consecutive hours ≥ EPA)

Each row is a distinct excursion where radon stayed at or above the EPA action level (4.0 pCi/L) for one or more consecutive hours. Each event is a discrete occurrence that the mitigation system did not prevent — they do not average out into the mean. Pay attention to *when* the events occurred relative to the mitigation phases, not just *how many* there are. (Note: per the data-quality notes above, peaks shown here may be underestimates due to window-opening during real-time observations.)

In [13]:
above = df_post['radon'] >= EPA_LEVEL
groups = (above != above.shift()).cumsum()
spike_events = []
for _, sub in df_post[above].groupby(groups[above]):
    spike_events.append({
        'start':  sub.index.min(),
        'end':    sub.index.max(),
        'hours':  len(sub),
        'peak':   sub['radon'].max(),
        'mean':   round(sub['radon'].mean(), 2),
    })
spike_events_df = pd.DataFrame(spike_events)
print(f'{len(spike_events_df)} distinct events >= EPA across {len(df_post)} hours ({len(df_post)/24:.1f} days) of post-move data')
spike_events_df

12 distinct events >= EPA across 532 hours (22.2 days) of post-move data


,start,end,hours,peak,mean
0,2026-05-14 20:29:00,2026-05-14 21:29:00,2,7.5,6.40
1,2026-05-15 10:29:00,2026-05-15 10:29:00,1,4.1,4.10
2,2026-05-15 13:29:00,2026-05-15 14:29:00,2,6.5,5.35
3,2026-05-16 12:29:00,2026-05-16 12:29:00,1,5.3,5.30
4,2026-05-16 21:29:00,2026-05-16 23:29:00,3,15.0,10.07
5,2026-05-17 10:29:00,2026-05-17 10:29:00,1,4.7,4.70
6,2026-05-24 23:31:00,2026-05-24 23:31:00,1,4.2,4.20
7,2026-05-25 20:31:00,2026-05-25 21:31:00,2,6.6,6.30
8,2026-05-26 20:31:00,2026-05-26 21:31:00,2,6.3,5.90
9,2026-06-02 19:31:00,2026-06-02 21:31:00,3,10.3,7.23


## Phase comparison

Three mitigation phases bounded by the May 18 and May 23 events. Phases are short (especially P1 at ~4 days), so treat differences as suggestive rather than conclusive.

In [14]:
def phase_stats(s):
    if len(s) == 0:
        return pd.Series({'days': 0, 'n': 0, 'mean': float('nan'),
                          'p50': float('nan'), 'p75': float('nan'),
                          'p90': float('nan'), 'p95': float('nan'),
                          'max': float('nan'),
                          'above_EPA': '0h (0.0%)',
                          'above_WHO': '0h (0.0%)'})
    span_days = (s.index.max() - s.index.min()).total_seconds() / 86400
    n_epa = int((s >= EPA_LEVEL).sum())
    n_who = int((s >= WHO_LEVEL).sum())
    return pd.Series({
        'days':       round(span_days, 1),
        'n':          len(s),
        'mean':       round(s.mean(), 2),
        'p50':        round(s.quantile(0.50), 2),
        'p75':        round(s.quantile(0.75), 2),
        'p90':        round(s.quantile(0.90), 2),
        'p95':        round(s.quantile(0.95), 2),
        'max':        round(s.max(), 2),
        'above_EPA':  f'{n_epa}h ({n_epa / len(s) * 100:.1f}%)',
        'above_WHO':  f'{n_who}h ({n_who / len(s) * 100:.1f}%)',
    })

phase_series = {}
for name, start, end in PHASES:
    start_ts = pd.Timestamp(start)
    end_ts   = pd.Timestamp(end) if end else (df.index.max() + pd.Timedelta(seconds=1))
    mask = (df.index >= start_ts) & (df.index < end_ts)
    phase_series[name] = df.loc[mask, 'radon']

summary = pd.DataFrame({name: phase_stats(s) for name, s in phase_series.items()}).T
summary

,days,n,mean,p50,p75,p90,p95,max,above_EPA,above_WHO
"P1: fan + 2.5"" WC",4.0,70,2.16,1.3,2.82,4.76,6.72,15.0,10h (14.3%),18h (25.7%)
"P2: 3.5"" WC + sealing",5.0,89,0.88,0.8,1.0,1.42,1.86,3.8,0h (0.0%),2h (2.2%)
P3: + ext. tweaks,20.4,373,0.94,0.6,0.8,1.3,2.82,14.0,12h (3.2%),20h (5.4%)


In [15]:
phase_long = pd.concat([
    s.to_frame('radon').assign(phase=name)
    for name, s in phase_series.items()
]).reset_index()

fig = px.box(phase_long, x='phase', y='radon', points='outliers',
             title='Radon distribution by mitigation phase',
             labels={'radon': 'pCi/L', 'phase': ''})
annotate(fig)
fig.show()

## Observations and remaining questions

**1. Seasonality — winter not yet observed.** Radon levels in this climate are typically highest in winter (Nov–Mar) due to closed-window operation and the stack effect actively drawing radon-laden air up from the garage and structure below. The data above covers late spring only, which represents a seasonal floor. Effective year-round performance can only be confirmed by observing the system through at least one full heating season.

**2. Window-opening during elevated readings.** Per the data-quality note in the introduction, when elevated readings were observed in real time, windows were opened to ventilate. This likely lowered some readings during those events, so the spike peaks and counts above are likely underestimates of the unmitigated indoor levels.

**3. Mean vs. distribution.** EPA's 4.0 pCi/L is an annualized-average benchmark, but a mitigation system's mechanical function is to prevent individual excursions of radon-laden air. A system whose mean is below 4.0 but whose 95th percentile or daily max still hits EPA is reducing typical exposure without fully controlling radon entry. Distribution metrics (percentiles, hours-above-threshold, individual spike events) belong alongside the mean when judging effectiveness.

**4. Spike events continue through the end of the record — and the worst is the most recent.** EPA-exceeding events keep occurring during P3 (the post-external-tweak period), and the single largest P3 excursion is the *final* reading in the dataset: a 2-hour event peaking at 14.0 pCi/L on the evening of June 12, the highest reading since the May 16 peak of 15.0. Earlier P3 spikes include a 3-hour, 10.3 pCi/L event on June 2 and a 2-hour, 7.6 pCi/L event on June 6. Far from tapering off, the most recent data contains the worst spike of the entire post-external-tweak phase. The spike-events table above lists each one with timestamps and durations.

**5. Evening risk window.** Spike events cluster between 19:00 and 22:00 — including the most recent and largest one, on the evening of June 12. The hour-of-day boxplot, evening-only timeline, and individual spike-event timestamps all converge on this pattern. The cause is not yet identified — candidate hypotheses include HVAC cycles, kitchen exhaust depressurization, or evening interior cool-down creating negative pressure relative to the garage below. Worth investigating before declaring the system stable.

**6. A better-performing configuration is already documented for this unit.** The temporary system that preceded the current one held radon to a median of 0.4 pCi/L (90th percentile ≈ 1.2; steady-operation maximum ≈ 2.7) in the 2023 commissioning data — staying below the EPA action level in normal operation. The current permanent system differs in one intended respect: the fan was relocated farther from the suction points (the vacuum was later raised from 2.5" to 3.5" WC to compensate). Its typical level and p90 now roughly match the prior system, but the repeated 4–15 pCi/L spikes seen above did not occur under it — which points to the current fan configuration, rather than an unsolvable site condition, as the most likely driver of the excursions. A known-achievable baseline exists in this same unit, and is the natural target for further work.